# E. coli FBA Analysis — Part 1
### Basic Flux Balance Analysis and Reaction Knockouts

---

This notebook is the computational equivalent of the EcoCyc Pathway Tools lab exercise.
Instead of the web-based EcoCyc interface, we use **COBRApy** — the standard Python library
for constraint-based metabolic modelling — together with **Escher** for pathway visualization.

**What you will do:**
1. Load a genome-scale metabolic model of *E. coli* K-12
2. Run FBA under aerobic glucose growth conditions
3. Identify the highest-flux reactions and visualize fluxes on a metabolic map
4. Knock out a specific reaction and compare the results
5. Answer two short analysis questions

**Background — Flux Balance Analysis (FBA)**

FBA finds the steady-state flux distribution that maximises a chosen objective (here: growth rate / biomass production) subject to:
- **Stoichiometric constraints** — mass balance at every metabolite node
- **Thermodynamic / capacity bounds** — each reaction flux $v_i$ is bounded: $lb_i \le v_i \le ub_i$

The result is a vector of reaction fluxes in units of **mmol g$_{DW}^{-1}$ h$^{-1}$**, and an optimal biomass flux in **h$^{-1}$**.

> **How to use this notebook:** Run each cell in order (Shift+Enter or the ▶ button).
> Cells marked **✏️ YOUR ANSWER** contain text fields for you to fill in.

---
## ⚙️ Setup — Run This First
*This cell installs the required packages. It takes ~1 minute and only needs to run once per Colab session.*

In [ ]:
# Install required packages
import subprocess, sys

print("Installing packages... (this takes about 60 seconds)")
subprocess.run([sys.executable, "-m", "pip", "install", "cobra", "escher",
                "pandas", "matplotlib", "seaborn", "--quiet"], check=True)

import cobra
import escher
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print(f"✅ COBRApy version: {cobra.__version__}")
print(f"✅ Escher version:  {escher.__version__}")
print("\nSetup complete — proceed to Section 1.")

---
## Section 1 — Load and Explore the *E. coli* Model

We use the **E. coli core metabolic model** (`e_coli_core`), a curated, textbook-quality
model of central carbon metabolism in *E. coli* K-12.
It contains 72 metabolites, 95 reactions, and 137 genes.

This is the COBRApy equivalent of loading the *"Glucose oxygen respiration"* model in EcoCyc.
The default medium is aerobic minimal medium with glucose as the sole carbon source.

In [ ]:
# Download and load the E. coli core model from BiGG Models
import urllib.request, os

MODEL_URL  = "http://bigg.ucsd.edu/static/models/e_coli_core.xml"
MODEL_FILE = "e_coli_core.xml"

if not os.path.exists(MODEL_FILE):
    print("Downloading model from BiGG Models...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_FILE)

model = cobra.io.read_sbml_model(MODEL_FILE)

print("=" * 55)
print(f"  Model loaded:   {model.id}")
print(f"  Reactions:      {len(model.reactions)}")
print(f"  Metabolites:    {len(model.metabolites)}")
print(f"  Genes:          {len(model.genes)}")
print(f"  Objective:      {list(model.objective.variables)[0].name}")
print("=" * 55)

In [ ]:
# Show the current growth medium (active exchange reaction bounds)
print("Active nutrient exchanges (uptake = negative flux):")
print("-" * 55)

medium_rows = []
for rxn_id, lb in model.medium.items():
    rxn = model.reactions.get_by_id(rxn_id)
    metabolite_name = list(rxn.metabolites)[0].name
    medium_rows.append({"Reaction ID": rxn_id, "Metabolite": metabolite_name,
                        "Max uptake (mmol/gDW/h)": lb})

medium_df = pd.DataFrame(medium_rows).sort_values("Metabolite")
display(medium_df.to_string(index=False))

print("\n📌  EX_glc__D_e (glucose) and EX_o2_e (oxygen) are both open")
print("    → this corresponds to aerobic growth on glucose.")

---
## Section 2 — Run FBA: Aerobic Growth on Glucose

We solve the FBA problem to maximise biomass production.
This corresponds to *Execute* in EcoCyc after loading the Glucose oxygen respiration model.

In [ ]:
# @title FBA method { run: "auto" }
# Choose between standard FBA and parsimonious FBA (pFBA).
# pFBA = standard FBA + a second optimisation that picks the solution
#        with the SMALLEST total flux among all optimal solutions.
# This removes thermodynamically-infeasible loops and gives cleaner,
# more biologically interpretable flux distributions.

USE_PFBA = True  # @param {type:"boolean"}

from cobra.flux_analysis import pfba

# Identify the biomass (objective) reaction once - we need its ID
# because under pFBA the `objective_value` is the total-flux sum, not biomass.
BIOMASS_RXN_ID = next(r.id for r in model.reactions if r.objective_coefficient != 0)
print(f"Biomass reaction: {BIOMASS_RXN_ID}")

def run_fba(mdl):
    """Run pFBA or standard FBA, depending on USE_PFBA. Returns a cobra Solution."""
    if USE_PFBA:
        try:
            return pfba(mdl)
        except Exception as e:
            print(f"  (pFBA failed: {e} -> falling back to standard FBA)")
            return mdl.optimize()
    else:
        return mdl.optimize()

# Run FBA - maximise biomass under aerobic glucose conditions
solution = run_fba(model)

print(f"\nFBA method:    {'Parsimonious FBA (pFBA)' if USE_PFBA else 'Standard FBA'}")
print(f"FBA status:    {solution.status}")
print(f"Biomass flux:  {solution.fluxes[BIOMASS_RXN_ID]:.4f} h⁻¹")


### 2a — Biomass Flux

In [ ]:
biomass_rxn = model.reactions.get_by_id(BIOMASS_RXN_ID)
biomass_flux = solution.fluxes[BIOMASS_RXN_ID]

print(f"Biomass reaction ID: {biomass_rxn.id}")
print(f"  Biomass flux (growth rate):  {biomass_flux:.4f} h⁻¹")
print()
print(f"This is the predicted specific growth rate (μ) of E. coli under this condition.")
print(f"At this rate, the doubling time would be ln(2) / {biomass_flux:.4f} ≈ {__import__('numpy').log(2)/biomass_flux*60:.1f} minutes.")


**✏️ YOUR ANSWER — 2a:**

*Record the biomass flux value here:*

> **Biomass flux = ________ h⁻¹**

### 2b — Reaction with the Highest Flux

In [ ]:
# Build a table of all non-zero, non-exchange fluxes
flux_df = solution.fluxes.reset_index()
flux_df.columns = ["Reaction ID", "Flux"]

# Annotate with reaction name, subsystem, and equation
rows = []
for _, row in flux_df.iterrows():
    rxn = model.reactions.get_by_id(row["Reaction ID"])
    rows.append({
        "Reaction ID":  rxn.id,
        "Name":         rxn.name,
        "Subsystem":    rxn.subsystem if rxn.subsystem else "—",
        "Flux":         row["Flux"],
        "|Flux|":       abs(row["Flux"]),
        "Equation":     rxn.reaction
    })

annot_df = pd.DataFrame(rows)

# Filter: active reactions only, exclude exchange/demand/sink reactions
internal = annot_df[
    (annot_df["|Flux|"] > 1e-6) &
    (~annot_df["Reaction ID"].str.startswith(("EX_", "DM_", "SK_", "BIOMASS")))
].sort_values("|Flux|", ascending=False)

print("Top 15 active internal reactions by |flux|:")
print("-" * 55)
display(internal[["Reaction ID", "Name", "Subsystem", "Flux"]].head(15).to_string(index=False))

In [ ]:
# Show the top reaction in detail
top_rxn_id   = internal.iloc[0]["Reaction ID"]
top_rxn_flux = internal.iloc[0]["Flux"]
top_rxn      = model.reactions.get_by_id(top_rxn_id)

print("=" * 65)
print(f"  Highest-flux reaction:  {top_rxn.name} ({top_rxn_id})")
print(f"  Flux:                   {top_rxn_flux:.4f} mmol gDW⁻¹ h⁻¹")
print(f"  Subsystem:              {top_rxn.subsystem}")
print(f"  Equation:               {top_rxn.reaction}")
print("=" * 65)

**✏️ YOUR ANSWER — 2b:**

*Record the reaction name, ID, flux value, and a brief interpretation:*

> **Reaction name (ID) = __________ (________)**  
> **Flux = __________ mmol gDW⁻¹ h⁻¹**
>
> *What does this reaction do?*  
> *(Your answer here)*

### 2c — Metabolic Map Visualization (Cellular Overview)

Escher renders an interactive map equivalent to the *"Show Fluxes on Cellular Overview"* view in EcoCyc.
Reaction arrows are coloured and scaled by flux magnitude.
Positive flux = forward direction; negative = reverse.

In [ ]:
from escher import Builder
from IPython.display import HTML

def show_escher_map(html_path, height=600):
    """Display a saved Escher map inline (works in Colab — avoids localhost)."""
    with open(html_path, 'r', encoding='utf-8') as f:
        content = f.read()
    srcdoc = content.replace('&', '&amp;').replace('"', '&quot;')
    return HTML(
        f'<iframe srcdoc="{srcdoc}" width="100%" height="{height}" '
        f'style="border:1px solid #ccc;"></iframe>'
    )

# Build the Escher flux map
builder = Builder(
    map_name      = "e_coli_core.Core metabolism",
    model_name    = "e_coli_core",
    reaction_data = dict(solution.fluxes),
    # Colour scale: blue (negative) -> white (zero) -> red (positive)
    reaction_scale = [
        {"type": "min",  "color": "#2166ac", "size": 10},
        {"type": "value", "value": 0, "color": "#f7f7f7", "size": 4},
        {"type": "max",  "color": "#d6604d", "size": 20},
    ],
    metabolism_data = None
)

# Save to HTML and display inline via srcdoc iframe (Colab-safe)
MAP_HTML = "flux_map_aerobic.html"
builder.save_html(MAP_HTML)

print(f"Flux map saved to: {MAP_HTML}")
print("Displaying interactive map (scroll/zoom to explore):")
show_escher_map(MAP_HTML)


**✏️ YOUR ANSWER — 2c:**

*Take a screenshot of the map (or use the camera icon in Colab) and paste it into your lab report.*

> **[Screenshot of aerobic glucose flux map]**

### 2d — Dashboard View (Energy & Cofactor Summary)

This chart mirrors the EcoCyc *"Show Fluxes on Dashboard"* view.
It summarises fluxes grouped by metabolic subsystem, with a special focus on
ATP and NADH production/consumption — the key energy currencies.

In [ ]:
def plot_dashboard(sol, mdl, title="Flux Dashboard"):
    """Produce subsystem bar chart + ATP/NADH balance panel."""
    fluxes = sol.fluxes

    # --- Panel 1: Subsystem fluxes ---
    subsystem_flux = {}
    for rxn in mdl.reactions:
        if rxn.id.startswith(("EX_", "DM_", "SK_", "BIOMASS")):
            continue
        sub = rxn.subsystem if rxn.subsystem else "Other"
        subsystem_flux.setdefault(sub, 0)
        subsystem_flux[sub] += abs(fluxes[rxn.id])

    sub_df = pd.DataFrame(list(subsystem_flux.items()),
                          columns=["Subsystem", "Total |Flux|"])
    sub_df = sub_df[sub_df["Total |Flux|"] > 0.5].sort_values(
        "Total |Flux|", ascending=True)

    # --- Panel 2: ATP-producing vs ATP-consuming reactions ---
    atp   = mdl.metabolites.get_by_id("atp_c")
    atp_prod, atp_cons = [], []
    for rxn in atp.reactions:
        coeff = rxn.metabolites[atp]
        net   = coeff * fluxes[rxn.id]
        label = rxn.name or rxn.id
        if net > 0.01:
            atp_prod.append((label[:28], net))
        elif net < -0.01:
            atp_cons.append((label[:28], abs(net)))

    atp_prod = sorted(atp_prod, key=lambda x: x[1], reverse=True)[:8]
    atp_cons = sorted(atp_cons, key=lambda x: x[1], reverse=True)[:8]

    # --- Panel 3: NADH balance ---
    nadh  = mdl.metabolites.get_by_id("nadh_c")
    nadh_prod, nadh_cons = [], []
    for rxn in nadh.reactions:
        coeff = rxn.metabolites[nadh]
        net   = coeff * fluxes[rxn.id]
        label = rxn.name or rxn.id
        if net > 0.01:
            nadh_prod.append((label[:28], net))
        elif net < -0.01:
            nadh_cons.append((label[:28], abs(net)))

    nadh_prod = sorted(nadh_prod, key=lambda x: x[1], reverse=True)[:8]
    nadh_cons = sorted(nadh_cons, key=lambda x: x[1], reverse=True)[:8]

    # --- Plot ---
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(title, fontsize=16, fontweight="bold", y=0.98)

    gs = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.35)

    # Subsystem bar
    ax0 = fig.add_subplot(gs[:, 0])
    colours = plt.cm.viridis(np.linspace(0.2, 0.85, len(sub_df)))
    bars = ax0.barh(sub_df["Subsystem"], sub_df["Total |Flux|"], color=colours)
    ax0.set_xlabel("Sum of |flux| (mmol gDW⁻¹ h⁻¹)", fontsize=10)
    ax0.set_title("Total Activity by Subsystem", fontsize=12, fontweight="bold")
    for bar, v in zip(bars, sub_df["Total |Flux|"]):
        ax0.text(v + 0.3, bar.get_y() + bar.get_height()/2,
                 f"{v:.1f}", va="center", fontsize=8)
    ax0.tick_params(axis="y", labelsize=9)

    # ATP producers
    ax1 = fig.add_subplot(gs[0, 1])
    if atp_prod:
        names, vals = zip(*atp_prod)
        ax1.barh(names, vals, color="#2ecc71")
    ax1.set_title("ATP Production", fontsize=11, fontweight="bold", color="#27ae60")
    ax1.set_xlabel("mmol gDW⁻¹ h⁻¹", fontsize=9)
    ax1.tick_params(axis="y", labelsize=8)

    # NADH producers
    ax2 = fig.add_subplot(gs[1, 1])
    if nadh_prod:
        names, vals = zip(*nadh_prod)
        ax2.barh(names, vals, color="#e74c3c")
    ax2.set_title("NADH Production", fontsize=11, fontweight="bold", color="#c0392b")
    ax2.set_xlabel("mmol gDW⁻¹ h⁻¹", fontsize=9)
    ax2.tick_params(axis="y", labelsize=8)

    plt.savefig(f"dashboard_{title.replace(' ', '_')}.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Dashboard saved as: dashboard_{title.replace(' ', '_')}.png")


plot_dashboard(solution, model, "Aerobic Glucose Growth")

**✏️ YOUR ANSWER — 2d:**

*The dashboard image is saved automatically. Paste it into your lab report and add a brief caption.*

> **[Dashboard screenshot — aerobic glucose]**

---
## Section 3 — Reaction Knockout Analysis

This section corresponds to **Step 3** of the lab:
*"Add the reaction identified in the list for exclusion"* (i.e. set its flux to zero).

### How to choose your reaction

Your instructor will assign you a specific reaction to knock out. You can also explore
the list below to understand what happens to growth under different genetic perturbations.

| Reaction ID | Reaction Name | Pathway |
|-------------|---------------|---------|
| `PGI`       | Phosphoglucose isomerase | Glycolysis |
| `PFK`       | Phosphofructokinase | Glycolysis |
| `FBA`       | Fructose-bisphosphate aldolase | Glycolysis |
| `TPI`       | Triosephosphate isomerase | Glycolysis |
| `G6PDH2r`   | Glucose-6-phosphate dehydrogenase | Pentose phosphate |
| `CS`        | Citrate synthase | TCA cycle |
| `ACONTa`    | Aconitase (half-reaction a) | TCA cycle |
| `ICDH`      | Isocitrate dehydrogenase | TCA cycle |
| `CYTBD`     | Cytochrome bd oxidase | Oxidative phosphorylation |
| `ATPS4r`    | ATP synthase | Oxidative phosphorylation |
| `PPC`       | Phosphoenolpyruvate carboxylase | Anaplerosis |
| `ME1`       | Malic enzyme (NAD) | Anaplerosis |

**Enter your assigned reaction ID in the cell below.**

In [ ]:
# @title Reaction Knockout Configuration { run: "auto" }
# ─────────────────────────────────────────────────────────
# Enter your assigned reaction ID (see table above):
REACTION_TO_KNOCKOUT = "PGI" # @param ["PGI", "PFK", "FBA", "TPI", "G6PDH2r", "CS", "ACONTa", "ICDH", "CYTBD", "ATPS4r", "PPC", "ME1"]
# ─────────────────────────────────────────────────────────

ko_rxn = model.reactions.get_by_id(REACTION_TO_KNOCKOUT)
print(f"Selected knockout:  {ko_rxn.name} ({REACTION_TO_KNOCKOUT})")
print(f"Subsystem:          {ko_rxn.subsystem}")
print(f"Equation:           {ko_rxn.reaction}")
print(f"Current bounds:     [{ko_rxn.lower_bound}, {ko_rxn.upper_bound}]")
print()
print("This reaction will be knocked out by constraining its flux to 0.")

### 3a — Run FBA with the Knockout

In [ ]:
# Use model.copy() so the original model is not modified
ko_model = model.copy()
ko_model.reactions.get_by_id(REACTION_TO_KNOCKOUT).knock_out()

ko_solution = run_fba(ko_model)

print(f"Knockout reaction:  {REACTION_TO_KNOCKOUT}")
print(f"FBA status:         {ko_solution.status}")
print()

wt_biomass = solution.fluxes[BIOMASS_RXN_ID]
ko_biomass = ko_solution.fluxes[BIOMASS_RXN_ID] if ko_solution.status == "optimal" else 0.0
delta      = ko_biomass - wt_biomass
pct_change = (delta / wt_biomass * 100) if wt_biomass > 0 else 0

print("=" * 55)
print(f"  WT biomass flux:  {wt_biomass:.4f} h⁻¹")
print(f"  KO biomass flux:  {ko_biomass:.4f} h⁻¹")
print(f"  Δ biomass:        {delta:+.4f} h⁻¹  ({pct_change:+.1f}%)")
print("=" * 55)

if ko_biomass < 1e-6:
    print("\n⚠️  Growth is completely abolished — this is an ESSENTIAL reaction.")
elif abs(pct_change) < 1:
    print("\n✅  Growth is essentially unchanged — this reaction is dispensable.")
else:
    print(f"\n⚠️  Growth is reduced by {abs(pct_change):.1f}% — this reaction is important but not essential.")

In [ ]:
# Show highest-flux reactions after knockout
if ko_solution.status == "optimal":
    ko_rows = []
    for rxn in ko_model.reactions:
        if rxn.id.startswith(("EX_", "DM_", "SK_", "BIOMASS")):
            continue
        f = ko_solution.fluxes[rxn.id]
        if abs(f) > 1e-6:
            ko_rows.append({"Reaction ID": rxn.id, "Name": rxn.name,
                            "Subsystem": rxn.subsystem or "—",
                            "Flux": f, "|Flux|": abs(f)})
    ko_flux_df = pd.DataFrame(ko_rows).sort_values("|Flux|", ascending=False)
    print(f"Top 15 active internal reactions after {REACTION_TO_KNOCKOUT} knockout:")
    display(ko_flux_df[["Reaction ID", "Name", "Subsystem", "Flux"]].head(15).to_string(index=False))
else:
    print("Model is infeasible — no flux solution exists after this knockout.")

In [ ]:
# Escher map for the knockout condition
if ko_solution.status == "optimal":
    ko_builder = Builder(
        map_name      = "e_coli_core.Core metabolism",
        model_name    = "e_coli_core",
        reaction_data = dict(ko_solution.fluxes),
        reaction_scale = [
            {"type": "min",  "color": "#2166ac", "size": 10},
            {"type": "value", "value": 0, "color": "#f7f7f7", "size": 4},
            {"type": "max",  "color": "#d6604d", "size": 20},
        ]
    )
    KO_MAP_HTML = f"flux_map_{REACTION_TO_KNOCKOUT}_knockout.html"
    ko_builder.save_html(KO_MAP_HTML)
    print(f"Knockout flux map saved: {KO_MAP_HTML}")
    display(show_escher_map(KO_MAP_HTML))
else:
    print("No flux map - model is infeasible after this knockout.")


In [ ]:
# Dashboard for the knockout condition
if ko_solution.status == "optimal":
    plot_dashboard(ko_solution, ko_model,
                   f"Knockout: {REACTION_TO_KNOCKOUT}")
else:
    print("No dashboard — model is infeasible after this knockout.")

**✏️ YOUR ANSWER — 3a:**

*Repeat the measurements from Section 2 for the knockout condition:*

> **Biomass flux (KO) = ________ h⁻¹** *(corresponds to 2a)*  
> **Highest-flux reaction (KO) = ________ (________) at ________ mmol gDW⁻¹ h⁻¹** *(corresponds to 2b)*  
> **[Screenshot: KO flux map]** *(corresponds to 2c)*  
> **[Screenshot: KO dashboard]** *(corresponds to 2d)*

---
## Section 4 — Comparison and Analysis Questions

Use the side-by-side comparison below to answer the two analysis questions.

In [ ]:
# Side-by-side summary
wt_sol  = solution
ko_sol  = ko_solution

print("=" * 65)
print(f"{'Metric':<40} {'Wild-type':>10} {'KO (' + REACTION_TO_KNOCKOUT + ')':>12}")
print("-" * 65)

# Biomass
ko_bm = ko_sol.fluxes[BIOMASS_RXN_ID] if ko_sol.status == "optimal" else 0.0
print(f"{'Biomass flux (h⁻¹)':<40} {wt_biomass:>10.4f} {ko_bm:>12.4f}")

# Key exchange fluxes
for ex_id, label in [("EX_glc__D_e", "Glucose uptake"),
                     ("EX_o2_e",     "O₂ uptake"),
                     ("EX_ac_e",     "Acetate secretion"),
                     ("EX_co2_e",    "CO₂ secretion"),
                     ("EX_etoh_e",   "Ethanol secretion"),
                     ("EX_for_e",    "Formate secretion")]:
    wt_v = wt_sol.fluxes.get(ex_id, 0)
    ko_v = ko_sol.fluxes.get(ex_id, 0) if ko_sol.status == "optimal" else 0
    if abs(wt_v) > 0.01 or abs(ko_v) > 0.01:
        print(f"{label:<40} {wt_v:>10.4f} {ko_v:>12.4f}")

print("=" * 65)
print("All fluxes in mmol gDW⁻¹ h⁻¹  (uptake = negative, secretion = positive)")

In [ ]:
# ATP production comparison — the main energy story
def atp_production_summary(sol, mdl, label):
    atp = mdl.metabolites.get_by_id("atp_c")
    total_prod = 0
    rows = []
    for rxn in atp.reactions:
        coeff = rxn.metabolites[atp]
        net = coeff * sol.fluxes[rxn.id]
        if net > 0.01:
            total_prod += net
            rows.append({"Reaction": rxn.name or rxn.id,
                         "Subsystem": rxn.subsystem or "—",
                         "ATP produced": net})
    df = pd.DataFrame(rows).sort_values("ATP produced", ascending=False)
    print(f"\n{label} — ATP production sources:")
    print(df.to_string(index=False))
    print(f"Total ATP production: {total_prod:.4f} mmol gDW⁻¹ h⁻¹")
    return df

wt_atp = atp_production_summary(wt_sol,  model,    "Wild-type")
if ko_sol.status == "optimal":
    ko_atp = atp_production_summary(ko_sol, ko_model, f"KO: {REACTION_TO_KNOCKOUT}")

In [ ]:
# Visual comparison: biomass flux bar + top flux changes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Wild-type vs {REACTION_TO_KNOCKOUT} Knockout",
             fontsize=14, fontweight="bold")

# Left: biomass comparison
ax = axes[0]
ko_bm_val = ko_sol.fluxes[BIOMASS_RXN_ID] if ko_sol.status == "optimal" else 0.0
bars = ax.bar(["Wild-type", f"ΔKO: {REACTION_TO_KNOCKOUT}"],
              [wt_biomass, ko_bm_val],
              color=["#2ecc71", "#e74c3c"], edgecolor="black", linewidth=0.8)
ax.set_ylabel("Biomass flux (h⁻¹)", fontsize=11)
ax.set_title("Growth Rate Comparison", fontsize=12)
ax.set_ylim(0, max(wt_biomass, ko_bm_val) * 1.25)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=10)

# Right: top 10 flux changes
ax2 = axes[1]
if ko_sol.status == "optimal":
    diff = {}
    for rxn in model.reactions:
        if rxn.id.startswith(("EX_", "DM_", "SK_", "BIOMASS")):
            continue
        d = ko_sol.fluxes[rxn.id] - wt_sol.fluxes[rxn.id]
        if abs(d) > 0.01:
            diff[rxn.name or rxn.id] = d

    diff_df = pd.DataFrame(list(diff.items()),
                           columns=["Reaction", "ΔFlux"])
    diff_df = diff_df.reindex(
        diff_df["ΔFlux"].abs().sort_values(ascending=False).index
    ).head(12)
    diff_df = diff_df.sort_values("ΔFlux")

    colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in diff_df["ΔFlux"]]
    ax2.barh(diff_df["Reaction"], diff_df["ΔFlux"], color=colors,
             edgecolor="black", linewidth=0.5)
    ax2.axvline(0, color="black", linewidth=0.8)
    ax2.set_xlabel("ΔFlux (KO − WT, mmol gDW⁻¹ h⁻¹)", fontsize=10)
    ax2.set_title("Top Flux Changes", fontsize=12)
    ax2.tick_params(axis="y", labelsize=8)
    patch_up   = mpatches.Patch(color="#2ecc71", label="Increased in KO")
    patch_down = mpatches.Patch(color="#e74c3c", label="Decreased in KO")
    ax2.legend(handles=[patch_up, patch_down], fontsize=9)
else:
    ax2.text(0.5, 0.5, "Infeasible — no growth",
             ha="center", va="center", fontsize=14, transform=ax2.transAxes)

plt.tight_layout()
COMPARE_IMG = f"comparison_{REACTION_TO_KNOCKOUT}_knockout.png"
plt.savefig(COMPARE_IMG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Comparison plot saved: {COMPARE_IMG}")

### Analysis Questions

Answer the questions below based on your results. Use the tables and figures generated in this notebook.

**Q1 — Did the biomass value change after the reaction knockout? If yes, by how much?**

*Make sure to include units and specify the direction of the change.*

> **Wild-type biomass = ________ h⁻¹**  
> **Knockout biomass  = ________ h⁻¹**  
> **Change = ________ h⁻¹ ( ________%)**
>
> *(Interpret the result — is the reaction essential? Partially required? Dispensable?)*  
> *(Your interpretation here)*

**Q2 — What differences do you observe in the energy production pathways?**

*Compare the ATP and NADH production sources in the wild-type vs knockout dashboards.*
*Refer to specific reactions or subsystems that changed.*

> *(Your answer here — use the ATP production comparison table and the dashboard screenshots)*

---
## Section 5 — Export Fluxes to iPath3 (Global Metabolic Map)

[**iPath3**](https://pathways.embl.de/) is an interactive KEGG-based metabolic map
that covers all known pathways across every organism. It is a great way to see
**where in the global metabolic network** your *E. coli* fluxes land.

The cell below converts your FBA results into the iPath3 selection format:

```
<KEGG_reaction_ID> W<width> <hex_color>
```

- **Width** is scaled by flux magnitude (thicker = higher flux)
- **Red** = forward flux, **Blue** = reverse flux
- Both wild-type and knockout results are exported

**How to use the output:**
1. Run the cell below — two text files are generated and previewed
2. Open https://pathways.embl.de/ and click the **Customize the appearance** (selection) box
3. Paste the contents of `ipath3_wt.txt` (or the knockout file) → click **Submit data**
4. The global metabolic map will highlight the reactions your model uses


In [ ]:
# Convert FBA fluxes to iPath3 selection format using KEGG reaction IDs
def fluxes_to_ipath3(sol, mdl, filename, threshold=1e-6, max_width=25):
    max_abs = max((abs(f) for f in sol.fluxes), default=1.0) or 1.0
    lines, n_exported, n_no_kegg = [], 0, 0
    for rxn_id, flux in sol.fluxes.items():
        if abs(flux) < threshold:
            continue
        rxn = mdl.reactions.get_by_id(rxn_id)
        kegg = rxn.annotation.get("kegg.reaction")
        if not kegg:
            n_no_kegg += 1
            continue
        kegg_ids = kegg if isinstance(kegg, list) else [kegg]
        width = max(3, int(round(max_width * abs(flux) / max_abs)))
        color = "#d6604d" if flux > 0 else "#2166ac"
        for kid in kegg_ids:
            lines.append(f"{kid} W{width} {color}")
            n_exported += 1
    text = "\n".join(lines)
    with open(filename, "w") as f:
        f.write(text)
    print(f"  -> Wrote {n_exported} entries to {filename} "
          f"(skipped {n_no_kegg} reactions without KEGG IDs)")
    return text

print("=" * 60)
print("WILD-TYPE aerobic growth -> iPath3")
print("=" * 60)
wt_ipath_text = fluxes_to_ipath3(solution, model, "ipath3_wt.txt")
print("\nPreview (first 10 lines):")
for line in wt_ipath_text.split("\n")[:10]:
    print("   " + line)

print("\n" + "=" * 60)
print(f"KNOCKOUT of {REACTION_TO_KNOCKOUT} -> iPath3")
print("=" * 60)
ko_file = f"ipath3_ko_{REACTION_TO_KNOCKOUT}.txt"
if ko_solution.status == "optimal":
    ko_ipath_text = fluxes_to_ipath3(ko_solution, ko_model, ko_file)
    print("\nPreview (first 10 lines):")
    for line in ko_ipath_text.split("\n")[:10]:
        print("   " + line)
else:
    print("  (skipped - knockout model was infeasible)")

# Offer direct downloads in Colab
print("\n" + "=" * 60)
print("DOWNLOAD your iPath3 files:")
print("=" * 60)
try:
    from google.colab import files
    files.download("ipath3_wt.txt")
    if ko_solution.status == "optimal":
        files.download(ko_file)
    print("  (files should appear in your browser's downloads)")
except Exception:
    print("  Not running in Colab - files saved to the working directory.")

print("\n>>> Next: open  https://pathways.embl.de/  and paste the file contents")
print("    into the 'Customize the appearance' selection box.")


---
## Section 6 — Explore Gene Essentiality (Optional)

COBRApy can systematically test every reaction for essentiality by knocking
each one out in turn and checking whether the model can still grow. Run the
cell below to identify all essential reactions in the *E. coli* core model.


In [ ]:
from cobra.flux_analysis import single_reaction_deletion

print("Running single-reaction deletion screen... (may take ~30 sec)")
deletion_results = single_reaction_deletion(model)

# Classify reactions
ESSENTIAL_THRESHOLD = 0.01  # growth fraction below which a reaction is "essential"

deletion_results["growth_fraction"] = deletion_results["growth"] / wt_biomass
deletion_results["status_label"] = deletion_results.apply(
    lambda row: "Essential" if row["growth"] < wt_biomass * ESSENTIAL_THRESHOLD
                else ("Reduced" if row["growth"] < wt_biomass * 0.99 else "Dispensable"),
    axis=1
)

print(f"\nTotal reactions screened: {len(deletion_results)}")
for label in ["Essential", "Reduced", "Dispensable"]:
    n = (deletion_results["status_label"] == label).sum()
    print(f"  {label:<12}: {n}")

print("\nEssential reactions (knockout abolishes growth):")
essential = deletion_results[deletion_results["status_label"] == "Essential"].copy()
ess_rows = []
for ids in essential["ids"]:
    rxn_id = list(ids)[0]
    try:
        rxn = model.reactions.get_by_id(rxn_id)
        ess_rows.append({"Reaction ID": rxn_id, "Name": rxn.name,
                         "Subsystem": rxn.subsystem or "—"})
    except Exception:
        pass

ess_df = pd.DataFrame(ess_rows)
display(ess_df.to_string(index=False))

---
## ✨ Bonus — pFBA vs Standard FBA on Your Knockout

By default this notebook uses **parsimonious FBA (pFBA)** because it removes
thermodynamically-infeasible flux loops and gives cleaner numbers.

**Standard FBA** finds *any* maximum-biomass solution — two different standard-FBA
runs can return wildly different individual reaction fluxes even though biomass is
the same. These extra "phantom" fluxes often show up as loops that simultaneously
produce and consume ATP.

The cell below re-runs your knockout using **plain FBA** (no parsimony) and
compares the two results. Things to look for:

- **Biomass flux** - should be identical (both methods maximise biomass)
- **Total |flux|** - should be ≥ for standard FBA (often much larger)
- **Per-reaction differences** - big differences on reactions in known
  thermodynamic cycles (e.g. transhydrogenase THD2, PPCK, MDH, FRD7)
  indicate loops that pFBA has pruned away

Change `USE_PFBA = False` at the top of Section 2, re-run the notebook from there,
and you'll see the same phenomenon propagate into the Escher maps, the iPath3
export, and the ATP-production table.


In [ ]:
# Re-run the knockout with standard (non-parsimonious) FBA and compare.
# This cell works regardless of the USE_PFBA setting above.

ko_model_fba = model.copy()
ko_model_fba.reactions.get_by_id(REACTION_TO_KNOCKOUT).bounds = (0, 0)
ko_sol_fba = ko_model_fba.optimize()    # plain FBA - no parsimony

def summarize(label, sol):
    if sol.status != "optimal":
        print(f"  {label:<30} infeasible")
        return
    bm  = sol.fluxes[BIOMASS_RXN_ID]
    o2  = sol.fluxes.get("EX_o2_e", 0.0)
    co2 = sol.fluxes.get("EX_co2_e", 0.0)
    atps= sol.fluxes.get("ATPS4r", 0.0)
    sumf= sol.fluxes.abs().sum()
    print(f"  {label:<30} biomass={bm:6.4f}  O2={o2:7.2f}  "
          f"CO2={co2:6.2f}  ATPS4r={atps:7.2f}  |Σflux|={sumf:7.1f}")

print(f"Knocked-out reaction: {REACTION_TO_KNOCKOUT}")
print("-" * 95)
summarize(f"{'pFBA' if USE_PFBA else 'Standard FBA'} (notebook default)", ko_solution)
summarize("Standard FBA (this cell)", ko_sol_fba)
print("-" * 95)

# Which reactions differ the most between the two solutions?
if ko_sol_fba.status == "optimal" and ko_solution.status == "optimal":
    diff = (ko_sol_fba.fluxes - ko_solution.fluxes).abs().sort_values(ascending=False)
    diff = diff[diff > 1e-4].head(10)
    if len(diff) == 0:
        print("\nThe two solutions are identical - your KO does not activate any loops.")
    else:
        print(f"\nTop {len(diff)} reactions where the two solutions differ most:")
        print(f"  {'Reaction':<12} {'|Δflux|':>10}   Name")
        for rxn_id, d in diff.items():
            name = ko_model_fba.reactions.get_by_id(rxn_id).name[:45]
            print(f"  {rxn_id:<12} {d:>10.3f}   {name}")
        print("\nReactions with large differences but identical biomass are typically")
        print("part of thermodynamically-infeasible ATP/NADH cycles that pFBA eliminates.")


---
## References

1. Orth, J.D., Fleming, R.M.T., and Palsson, B.Ø. (2010). [Reconstruction and Use of Microbial Metabolic Networks: the Core Escherichia coli Metabolic Model as an Educational Guide](https://dx.doi.org/10.1128/ecosalplus.10.2.1). *EcoSal Plus*.
2. Ebrahim, A., Lerman, J.A., Palsson, B.Ø., and Hyduke, D.R. (2013). [COBRApy: COnstraints-Based Reconstruction and Analysis for Python](https://dx.doi.org/10.1186/1752-0509-7-74). *BMC Systems Biology*, 7:74.
3. King, Z.A. *et al.* (2015). [Escher: A Web Application for Building, Sharing, and Embedding Data-Rich Visualizations of Biological Pathways](https://doi.org/10.1371/journal.pcbi.1004321). *PLOS Computational Biology*, 11(8):e1004321.
4. BiGG Models Database: [http://bigg.ucsd.edu](http://bigg.ucsd.edu)
5. EcoCyc: [https://ecocyc.org](https://ecocyc.org)

---
*Notebook developed for use in Experimental Synthetic Biology.*  
*Please direct questions to your instructor.*